In [2]:
import pandas as pd
import shutil

POOL_FILE = "/Users/tommasomilanino/Developer/THESIS/phase3_internals/dataset_building/prompt_pool.parquet"
AUG_FILE  = "/Users/tommasomilanino/Developer/THESIS/phase3_internals/dataset_building/augmented_languages.csv"

# Backup
shutil.copy(POOL_FILE, POOL_FILE.replace(".parquet", "_backup.parquet"))

pool = pd.read_parquet(POOL_FILE)
aug = pd.read_csv(AUG_FILE)

print(f"Pool: {len(pool)}   Augmented: {len(aug)}")
print(f"Colonne mancanti in aug: {set(pool.columns) - set(aug.columns)}")

Pool: 25858   Augmented: 2428
Colonne mancanti in aug: set()


In [3]:
merged = pd.concat([pool, aug[pool.columns]], ignore_index=True)

# Sanity check: nessun duplicato introdotto dal merge
dupes = merged["prompt_id"].duplicated().sum()
print(f"Duplicati per prompt_id: {dupes}")

if dupes > 0:
    merged = merged.drop_duplicates(subset="prompt_id", keep="first").reset_index(drop=True)
    print(f"Rimossi, restano {len(merged)} righe")

merged.to_parquet(POOL_FILE, engine="pyarrow")

print(f"\nPool finale: {len(merged)} prompt")
print(merged["language"].value_counts().to_string())
print()
print(merged["source"].value_counts().to_string())

Duplicati per prompt_id: 0

Pool finale: 28286 prompt
language
english    12246
russian     5000
arabic      4063
german      2977
french      2000
spanish     2000

source
ai_generated                    6597
chapter2_native_multilingual    5951
aya_redteaming                  4398
multilingual_safety_en          2788
augmented                       2428
wildjailbreak                   2000
chapter2_translated             1238
chapter2_ai_generated           1065
do_not_answer                    935
xstest                           445
advbench                         348
harmbench                         93
